In [ ]:
%%writefile program.cpp
#include <iostream>
#include <vector>
#include <omp.h>
#include <cstdlib>
#include <algorithm>

using namespace std;

//////////////////// SEQUENTIAL BUBBLE SORT ////////////////////
void sequentialBubbleSort(vector<int> arr) {
    int n = arr.size();

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n - 1; j++) {
            if (arr[j] > arr[j + 1]) {
                swap(arr[j], arr[j + 1]);
            }
        }
    }
}

//////////////////// PARALLEL BUBBLE SORT ////////////////////
void parallelBubbleSort(vector<int> &arr) {
    int n = arr.size();

    for (int i = 0; i < n; i++) {

        #pragma omp parallel for
        for (int j = i % 2; j < n - 1; j += 2) {

            if (arr[j] > arr[j + 1]) {
                swap(arr[j], arr[j + 1]);
            }
        }
    }
}

//////////////////// SEQUENTIAL MERGE SORT ////////////////////
void merge(vector<int> &arr, int l, int m, int r) {
    vector<int> temp(r - l + 1);

    int i = l, j = m + 1, k = 0;

    while (i <= m && j <= r) {
        if (arr[i] <= arr[j])
            temp[k++] = arr[i++];
        else
            temp[k++] = arr[j++];
    }

    while (i <= m) temp[k++] = arr[i++];
    while (j <= r) temp[k++] = arr[j++];

    for (int x = 0; x < k; x++)
        arr[l + x] = temp[x];
}

void sequentialMergeSort(vector<int> &arr, int l, int r) {
    if (l < r) {
        int m = (l + r) / 2;

        sequentialMergeSort(arr, l, m);
        sequentialMergeSort(arr, m + 1, r);

        merge(arr, l, m, r);
    }
}

//////////////////// PARALLEL MERGE SORT ////////////////////
void parallelMergeSort(vector<int> &arr, int l, int r) {

    if (l < r) {

        int m = (l + r) / 2;

        #pragma omp parallel sections
        {
            #pragma omp section
            parallelMergeSort(arr, l, m);

            #pragma omp section
            parallelMergeSort(arr, m + 1, r);
        }

        merge(arr, l, m, r);
    }
}

//////////////////// MAIN ////////////////////
int main() {

    int n = 10000;
    vector<int> arr(n);

    for (int i = 0; i < n; i++) {
        arr[i] = rand() % 1000;
    }

    vector<int> a1 = arr, a2 = arr, a3 = arr, a4 = arr;

    double start, end;

    //////////////////// SEQUENTIAL BUBBLE ////////////////////
    start = omp_get_wtime();
    sequentialBubbleSort(a1);
    end = omp_get_wtime();
    cout << "Sequential Bubble Sort Time: " << (end - start) << " sec\n";

    //////////////////// PARALLEL BUBBLE ////////////////////
    start = omp_get_wtime();
    parallelBubbleSort(a2);
    end = omp_get_wtime();
    cout << "Parallel Bubble Sort Time: " << (end - start) << " sec\n";

    //////////////////// SEQUENTIAL MERGE ////////////////////
    start = omp_get_wtime();
    sequentialMergeSort(a3, 0, n - 1);
    end = omp_get_wtime();
    cout << "Sequential Merge Sort Time: " << (end - start) << " sec\n";

    //////////////////// PARALLEL MERGE ////////////////////
    start = omp_get_wtime();
    parallelMergeSort(a4, 0, n - 1);
    end = omp_get_wtime();
    cout << "Parallel Merge Sort Time: " << (end - start) << " sec\n";

    //////////////////// SAMPLE OUTPUT ////////////////////
    cout << "\nFirst 10 sorted elements (Merge Sort):\n";
    for (int i = 0; i < 10; i++) {
        cout << a4[i] << " ";
    }

    cout << endl;

    return 0;
}


command to compile
!g++ -fopenmp program.cpp -o program

command to run
!./program